# Sales Funnel Optimization & Revenue Leakage Analysis

## Objective
Analyze customer movement through the sales funnel, identify drop-offs, quantify revenue leakage, and simulate optimization strategies.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)

In [ ]:
# Load data
df = pd.read_csv('../data/full2.csv')
df.head()

In [ ]:
# Data inspection
df.info()
df.describe()

In [ ]:
# Data cleaning
df['Order Date'] = pd.to_datetime(df['Order Date'], errors='coerce')
df['Expected Delivery Date'] = pd.to_datetime(df['Expected Delivery Date'], errors='coerce')
df['Discount'] = df['Discount'].fillna(0)

stage_order = ['Add_to_cart', 'Checkout', 'Purchase']
df['event_type'] = pd.Categorical(df['event_type'], categories=stage_order, ordered=True)

df = df[(df['Final Price'] > 0) & (df['Discount'] < 1)]
df.head()

In [ ]:
# Funnel size analysis
funnel_counts = (
    df.groupby('event_type')['customer_id']
      .nunique()
      .sort_index()
)
funnel_counts

In [ ]:
# Conversion rates
conversion_rates = funnel_counts / funnel_counts.shift(1)
conversion_rates

In [ ]:
# Funnel visualization
plt.figure()
funnel_counts.plot(kind='bar')
plt.title('Sales Funnel – Customers per Stage')
plt.xlabel('Stage')
plt.ylabel('Unique Customers')
plt.tight_layout()
plt.show()

In [ ]:
# Revenue leakage analysis
df['expected_price'] = df['List Price']
df['revenue_loss'] = df['expected_price'] - df['Final Price']

leakage_by_rep = (
    df.groupby('sales_rep')['revenue_loss']
      .sum()
      .sort_values(ascending=False)
)
leakage_by_rep

In [ ]:
plt.figure()
leakage_by_rep.plot(kind='bar')
plt.title('Revenue Leakage by Sales Rep')
plt.xlabel('Sales Rep')
plt.ylabel('Revenue Lost')
plt.tight_layout()
plt.show()

In [ ]:
# Lost revenue from dropped deals
lost_deals = df[df['deal_status'] == 'Lost']
lost_revenue = lost_deals['Final Price'].sum()
lost_revenue

In [ ]:
# Deal delay analysis
df = df.sort_values(['customer_id', 'Order Date'])
df['delay_days'] = (df['Expected Delivery Date'] - df['Order Date']).dt.days

avg_delay = df.groupby('event_type')['delay_days'].mean()
avg_delay

In [ ]:
plt.figure()
avg_delay.plot(kind='bar')
plt.title('Average Delay Between Funnel Stages')
plt.xlabel('Funnel Stage')
plt.ylabel('Average Delay (Days)')
plt.tight_layout()
plt.show()

In [ ]:
# Optimization scenarios
threshold = np.percentile(df['Discount'], 90)
df['high_discount_flag'] = np.where(df['Discount'] > threshold, 1, 0)

discount_summary = df.groupby('high_discount_flag')['revenue_loss'].sum()
discount_summary.index = ['Normal Discounts', 'High Discounts']
discount_summary

In [ ]:
plt.figure()
discount_summary.plot(kind='bar')
plt.title('Revenue Loss from High vs Normal Discounts')
plt.tight_layout()
plt.show()

In [ ]:
# Revenue recovery simulation
df['optimized_price'] = df['expected_price'] * 1.05
df['recovered_revenue'] = (df['optimized_price'] - df['Final Price']).clip(lower=0)

price_summary = pd.DataFrame({
    'Metric': ['Final Revenue', 'Optimized Revenue', 'Recovered Revenue'],
    'Amount': [
        df['Final Price'].sum(),
        df['optimized_price'].sum(),
        df['recovered_revenue'].sum()
    ]
})
price_summary

## Key Takeaways
- Identified major funnel drop-off points
- Quantified revenue leakage due to discounts and lost deals
- Simulated pricing optimization to estimate recoverable revenue

This notebook demonstrates an end-to-end business analytics workflow.